# 🎓 LouisFarm — Semaine 2 : Data Wrangling & Data Quality
## Dataset : Mobile Money Sénégal (5 100 transactions)

**Objectif :** Transformer un dataset brut imparfait en dataset analytique fiable, documenté et reproductible.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_mobilemoney_senegal

df_raw = gen_mobilemoney_senegal(n=5000)
print(f"Dataset chargé : {df_raw.shape}")
print(df_raw.head(4))

## Leçon 2.1 — Data Profiling Systématique

Avant de nettoyer, il faut **mesurer**. Le data profiling évalue 5 dimensions : Complétude, Cohérence, Validité, Unicité, Fraîcheur.

In [ ]:
# ─── DATA PROFILING COMPLET ────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════╗")
print("║        RAPPORT DE DATA PROFILING                 ║")
print("║        Mobile Money Sénégal 2022-2023            ║")
print("╚══════════════════════════════════════════════════╝\n")

n = len(df_raw)
print(f"Volume : {n:,} observations × {df_raw.shape[1]} variables\n")

# 1. Complétude
print("1. COMPLÉTUDE (valeurs manquantes)")
print("-" * 45)
missing = df_raw.isnull().sum()
for col in df_raw.columns:
    n_miss = missing[col]
    pct = n_miss/n*100
    bar = "█" * int(pct*0.5)
    print(f"  {col:<20} : {n_miss:>5} ({pct:>5.1f}%) {bar}")

# 2. Unicité (doublons)
print(f"\n2. UNICITÉ")
n_dup = df_raw.duplicated().sum()
n_dup_id = df_raw.duplicated(subset='id_transaction').sum()
print(f"  Lignes dupliquées totales  : {n_dup} ({n_dup/n*100:.1f}%)")
print(f"  IDs transaction dupliqués  : {n_dup_id} ({n_dup_id/n*100:.1f}%)")

# 3. Cohérence
print(f"\n3. COHÉRENCE — Valeurs de 'ville'")
print(df_raw['ville'].value_counts().to_string())

# 4. Types
print(f"\n4. TYPES DE DONNÉES")
for col in df_raw.columns:
    print(f"  {col:<22}: {str(df_raw[col].dtype):<12} | ex: {str(df_raw[col].dropna().iloc[0])[:25]}")


## Leçon 2.2 — Traitement des Valeurs Manquantes

In [ ]:
# ─── STRATÉGIES DE TRAITEMENT DES MANQUANTS ──────────────────────────────
df = df_raw.copy()
print("AVANT nettoyage :", df.isnull().sum().sum(), "valeurs manquantes\n")

# Stratégie 1 : Imputation par mode (genre_client)
mode_genre = df['genre_client'].mode()[0]
df['genre_client'].fillna(mode_genre, inplace=True)
print(f"genre_client : imputé par mode ('{mode_genre}')")

# Stratégie 2 : Imputation par médiane (age_client)
median_age = df['age_client'].median()
df['age_client'].fillna(median_age, inplace=True)
print(f"age_client   : imputé par médiane ({median_age:.0f} ans)")

print(f"\nAPRÈS imputation : {df.isnull().sum().sum()} valeurs manquantes")


## Leçon 2.3 — Nettoyage des Types et Cohérence

In [ ]:
# ─── STANDARDISATION DES VILLES ────────────────────────────────────────────
print("Villes AVANT standardisation:")
print(df['ville'].value_counts())

df['ville'] = df['ville'].str.strip().str.title()
print("\nVilles APRÈS standardisation:")
print(df['ville'].value_counts())

# ─── SUPPRESSION DES DOUBLONS ────────────────────────────────────────────────
print(f"\nLignes avant dédoublonnage : {len(df):,}")
df.drop_duplicates(subset='id_transaction', keep='first', inplace=True)
print(f"Lignes après dédoublonnage : {len(df):,}")
df.reset_index(drop=True, inplace=True)


In [ ]:
# ─── CONVERSION DES TYPES ───────────────────────────────────────────────────
# age_client → entier
df['age_client'] = df['age_client'].astype('Int64')  # Int64 nullable : accepte les NaN

# Parsing des dates (format mixte)
df['date'] = pd.to_datetime(df['date'], dayfirst=False, errors='coerce')
df['annee'] = df['date'].dt.year
df['mois'] = df['date'].dt.month
df['trimestre'] = df['date'].dt.quarter

print("Dates parsées ✅")
print(df[['date','annee','mois','trimestre']].head(5))
print(f"\nDates valides : {df['date'].notna().sum():,} / {len(df):,}")


## Leçon 2.4 — Détection et Traitement des Outliers

In [ ]:
# ─── DÉTECTION DES OUTLIERS PAR IQR ────────────────────────────────────────
Q1 = df['montant_xof'].quantile(0.25)
Q3 = df['montant_xof'].quantile(0.75)
IQR = Q3 - Q1
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

outliers = df[(df['montant_xof'] < borne_inf) | (df['montant_xof'] > borne_sup)]
print(f"Montants — Q1:{Q1:>10,.0f} XOF | Q3:{Q3:>10,.0f} XOF | IQR:{IQR:>10,.0f}")
print(f"Bornes   — Inf:{borne_inf:>9,.0f} XOF | Sup:{borne_sup:>9,.0f} XOF")
print(f"Outliers : {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1,2, figsize=(12,4))
fig.suptitle("Semaine 2 — Détection des outliers (Mobile Money Sénégal)", fontweight='bold')
axes[0].boxplot(df['montant_xof'].dropna())
axes[0].set_title("Boxplot montants (avec outliers)")
axes[0].set_ylabel("Montant (XOF)")

df_clean_amt = df[(df['montant_xof'] >= borne_inf) & (df['montant_xof'] <= borne_sup)]
axes[1].boxplot(df_clean_amt['montant_xof'])
axes[1].set_title("Boxplot montants (sans outliers)")
plt.tight_layout()
plt.savefig('./s2_outliers.png', dpi=100, bbox_inches='tight')
plt.show()


## Leçon 2.5 — Agrégation et Reshaping

In [ ]:
# ─── AGGREGATION AVANCÉE ────────────────────────────────────────────────────
print("Transactions par ville et type (pivot table):")
pivot = pd.pivot_table(df, values='montant_xof',
                       index='ville', columns='type_transaction',
                       aggfunc='sum', fill_value=0)
print((pivot/1e6).round(1))
print("(en millions XOF)")

# Merge avec données de population fictives
pop_data = pd.DataFrame({
    'ville': ['Dakar','Thiès','Kaolack','Ziguinchor','Saint-Louis','Diourbel','Tambacounda'],
    'population_2023': [3700000, 950000, 550000, 280000, 450000, 380000, 190000]
})
df_merged = df.groupby('ville')['montant_xof'].sum().reset_index()
df_merged = df_merged.merge(pop_data, on='ville', how='left')
df_merged['montant_par_habitant'] = df_merged['montant_xof'] / df_merged['population_2023']
df_merged = df_merged.sort_values('montant_par_habitant', ascending=False)
print("\nMontant de transactions par habitant :")
print(df_merged[['ville','montant_par_habitant']].round(0))


In [ ]:
# ─── DATA QUALITY REPORT ─────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════╗")
print("║                 DATA QUALITY REPORT                      ║")
print("║     Mobile Money Sénégal — Dataset Nettoyé              ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Lignes initiales        : {len(df_raw):>8,}                     ║")
print(f"║  Lignes après nettoyage  : {len(df):>8,}                     ║")
print(f"║  Doublons supprimés      : {len(df_raw)-len(df):>8,}                     ║")
print(f"║  Valeurs imputées        : genre+age                    ║")
print(f"║  Villes standardisées    : .title() appliqué            ║")
print(f"║  Colonnes ajoutées       : annee, mois, trimestre       ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Valeurs manquantes restantes : {df.isnull().sum().sum():>3}                    ║")
print(f"║  Statut QUALITÉ : ✅ PRÊT POUR ANALYSE                  ║")
print("╚══════════════════════════════════════════════════════════╝")

df.to_csv('./senegal_mobilemoney_clean.csv', index=False)
print("\nDataset propre sauvegardé ✅")


## 🧪 EXERCICES — Semaine 2

In [ ]:
# EXERCICE 2.1 — Créer une fonction de nettoyage réutilisable (★★☆)
def clean_dataset_louisfarm(df_input, drop_dup_cols=None, impute_cols=None):
    """
    Fonction générique de nettoyage pour les datasets LouisFarm.
    
    Paramètres:
        df_input     : DataFrame brut
        drop_dup_cols: Colonnes pour détecter les doublons (None = toutes)
        impute_cols  : dict {colonne: stratégie} où stratégie in ['mean','median','mode']
    
    Retourne:
        df_clean, rapport (dict)
    """
    df = df_input.copy()
    rapport = {'initial_rows': len(df)}
    
    # 1. Doublons
    n_before = len(df)
    df.drop_duplicates(subset=drop_dup_cols, keep='first', inplace=True)
    rapport['duplicates_removed'] = n_before - len(df)
    
    # 2. Imputation
    rapport['imputed'] = {}
    if impute_cols:
        for col, strategy in impute_cols.items():
            if col in df.columns:
                n_miss = df[col].isnull().sum()
                if strategy == 'mean': val = df[col].mean()
                elif strategy == 'median': val = df[col].median()
                elif strategy == 'mode': val = df[col].mode()[0]
                df[col].fillna(val, inplace=True)
                rapport['imputed'][col] = {'strategy': strategy, 'value': round(val,2) if isinstance(val,(int,float)) else val, 'n_filled': int(n_miss)}
    
    rapport['final_rows'] = len(df)
    rapport['missing_remaining'] = int(df.isnull().sum().sum())
    return df, rapport

# Test de la fonction
df_test = gen_mobilemoney_senegal(n=1000, seed=99)
df_cleaned, rap = clean_dataset_louisfarm(
    df_test, 
    drop_dup_cols='id_transaction',
    impute_cols={'genre_client': 'mode', 'age_client': 'median'}
)
print("Rapport de nettoyage :")
for k, v in rap.items():
    print(f"  {k}: {v}")
